In [3]:
import pandas as pd
from typing import Optional, List
from pydantic import BaseModel, Field
import re

In [4]:
# Load the uploaded Excel file to inspect its content
file_path = '../Ontology3.xlsx'
excel_data = pd.ExcelFile(file_path)

# Display the sheet names to understand the structure of the file
excel_data.sheet_names

['Sheet1']

In [5]:
# Load the sheet to inspect its content
gearbox_v3_df = excel_data.parse('Sheet1')
excel_data.close
# Display the first few rows of the dataframe to understand its structure
gearbox_v3_df.head()

,Parameter Name,Type,Description,Examples,MaxLength,Enum
0,age_min,integer,What is the minimum age of eligibility (in yea...,NaN,10.0,NaN
1,age_max,integer,What is the maximum age of eligibility (in yea...,NaN,NaN,NaN
2,weight_min,number,What is the minimum weight a patient may have ...,NaN,10.0,NaN
3,weight_max,number,What is the maximum weight a patient may have ...,NaN,10.0,NaN
4,sex,string,Which sexes are eligible in the eligibility cr...,NaN,15.0,"""Male"", ""Female"""


In [8]:
# Function to generate class fields and enums
def generate_class_fields_and_enums(df):
    fields = {}
    enum_classes = {}  # Dictionary to store generated enum classes as strings

    for _, row in df.iterrows():
        param_name = row['Parameter Name']
        
        # Determine the field type based on 'Type' and 'Enum' columns
        if row['Type'] in ['integer', 'number']:
            field_type = 'Optional[int]'
        elif pd.notna(row['Enum']) and 'Yes' in row['Enum']:
            field_type = 'Optional[Yes_no]'  # Assuming YesOrNo is a defined type elsewhere
        elif pd.notna(row['Enum']):
            field_type = f"Optional[List[{param_name.capitalize()}]]"
            
            # Create an enum class for this parameter
            enum_items = [item.strip().strip('"') for item in row['Enum'].split(',')]
            enum_class_str = f"class {param_name.capitalize()}(Enum):\n"
            for item in enum_items:
                if "(" in item and ")" in item:
                    enum_name = re.search(r"\((.*?)\)", item).group(1)
                    enum_name = enum_name.strip().replace(" ", "")
                    enum_name = enum_name.strip().replace("-", "").upper()
                else:
                    enum_name = item.strip().replace("-", "").replace(" ", "").upper()
                enum_class_str += f"    {enum_name} = \"{item}\"\n"
            
            enum_classes[param_name.capitalize()] = enum_class_str
            
        else:
            field_type = 'Optional[str]'
            

        # Combine description and examples if available
        description = row['Description']
        example = row['Examples']
        if pd.notna(example):
            description += f" Example: {example}"
        
        fields[param_name] = f"{field_type} = Field(\n        None, \n      description='{description}')"
    
    enum_items = ["Yes","No"]
    enum_class_str = f"class Yes_no(Enum):\n"
    for item in enum_items:
        enum_name = item.upper()
        enum_class_str += f"    {enum_name} = \"{item}\"\n"
    enum_classes["Yes_no"] = enum_class_str

    return fields, enum_classes


# Generate class fields from the DataFrame
class_fields, enum_classes = generate_class_fields_and_enums(gearbox_v3_df)

class_definition = "from enum import Enum\nfrom pydantic import BaseModel, Field\nfrom typing import Optional, List\n\n"
for enum_name, enum_definition in enum_classes.items():
    class_definition += f"{enum_definition}\n"

# Generating the class as a string
class_definition += "class Entity(BaseModel):\n"
for field_name, field_definition in class_fields.items():
    class_definition += f"    {field_name}: {field_definition}\n"

class_definition

'from enum import Enum\nfrom pydantic import BaseModel, Field\nfrom typing import Optional, List\n\nclass Sex(Enum):\n    MALE = "Male"\n    FEMALE = "Female"\n\nclass Diagnosis(Enum):\n    APL = "Acute promyelocytic leukemia (APL)"\n    AML = "Acute myeloid leukemia (AML)"\n    DSAML = "Down syndrome acute myeloid leukemia (DS AML)"\n    BALL = "B-cell acute lymphoblastic leukemia (B-ALL)"\n    JMML = "Juvenile myelomonocytic leukemia (JMML)"\n    MPAL = "Mixed-phenotype acute leukemia (MPAL)"\n    MDS = "Myelodysplastic syndromes (MDS)"\n    MDS = "Myelodysplastic syndrome (MDS) who have progressed to AML"\n    TAML = "Treatment-related acute myeloid leukemia (tAML)"\n    TALL = "T-cell acute lymphoblastic leukemia (T-ALL)"\n    ALL = "Acute lymphoblastic leukemia (ALL)"\n    KMT2AALL = "KMT2A acute lymphoblastic leukemia (KMT2A-ALL)"\n    ETPALL = "Early T-cell precursor acute lymphoblastic leukemia (ETP-ALL)"\n    ALAL = "Ambiguous lineage acute leukemia (ALAL)"\n\nclass Bm_recent_

In [10]:
# Define the file path to save the class definition
output_file_path = "Entity.py"

# Save the class definition to a file
with open(output_file_path, 'w') as file:
    file.write(class_definition)